![Astrofisica Computacional](../../new_logo.png)

# Computational Astrophysics –  ML Decision Trees III (Cross-validation of Decision Trees)

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---
Taken from previous lectures of this course.


En los notebooks anteriores, utilizamos la mediana de los residuos para evaluar las predicciones del modelo. Este es un método muy básico que se basa en la división del conjunto de datos original en subconjuntos de entrenamiento y prueba, y se denomina **validación por retención**. Una desventaja de este procedimiento es que la precisión obtenida depende de cómo se definen los subconjuntos.

Ahora, introduciremos un método de evaluación mejor llamado **validación cruzada k-fold**. Es similar a la validación por retención, pero el conjunto de datos original se dividirá en k subconjuntos y el modelo se entrenará k veces utilizando diferentes combinaciones de los subconjuntos, calculando la precisión en cada iteración (es decir, se calculará una validación por retención k veces).

Durante cada iteración, utilizaremos una combinación diferente de k-1 subconjuntos para entrenar el algoritmo y el subconjunto restante se utilizará para probar el modelo. Luego, calculamos la media de la precisión de las k mediciones para obtener una precisión global del modelo.

### El conjunto de datos

Una vez más, utilizaremos el archivo `sdss_galaxy_colors.npy`, pero esta vez consideraremos la característica `spec_class`. Esta característica indica si la muestra corresponde a un objeto cuasiestelar (QSO) o a una galaxia normal (GALAXY). Como es bien sabido, los QSO son galaxias con un núcleo activo (AGN), lo que implica un mayor brillo y, por lo tanto, una mejor detección con los instrumentos del SDSS, incluso para mayores desplazamientos al rojo.


In [ ]:
import numpy as np

data = np.load('sdss_galaxy_colors.npy')
data

The features in the data set are

| dtype | Feature|
|:-:|:-:|
|`u` |u band filter|
|`g` |g band filter|
|`r` |r band filter|
|`i` |i band filter|
|`z` |z band filter|
|`spec_class` |spectral class|
|`redshift` |redshift|
|`redshift_err` |redshift error|


In [ ]:
n = data.size
n

In [1]:
# Función que devuelve los 4 índices de color y los corrimientos al rojo.

# Crear los arrays con los cuatro indices 
# features = (nsize, 4)
# target = array redshifts
# classes = array spec_class


### Validación cruzada K-fold

Para implementar la validación cruzada K-fold, utilizaremos la función [sklearn.model_selection.KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) para dividir los conjuntos. El comando para realizar la división es:

```
kf = KFold(n_splits=k, shuffle=True)

```

donde `n_splits=k` define el número de subconjuntos y el argumento `shuffle` tiene el valor predeterminado 'False', pero es recomendable activarlo para obtener una distribución aleatoria de los elementos en cada subconjunto (esta distribución evita la aparición de sesgos debidos al orden de las muestras en el archivo de datos).

En este ejemplo, utilizaremos la función `KFold` con un valor de **k=5** y el árbol de decisión tendrá una profundidad de `max_depth=19` (de acuerdo con la discusión anterior sobre el sobreajuste).

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold

# kf con n_splits=5, shuffle True
# arbol con max_depth 19


El método `KFold.split()` se aplica al conjunto de características para generar los subconjuntos de entrenamiento y prueba. **Tenga en cuenta que este método solo define el conjunto de índices de cada subconjunto, pero no el conjunto en sí**. Por lo tanto, necesitamos usar estos índices para definir los subconjuntos que entrenarán y evaluarán las predicciones del algoritmo. Para implementar este procedimiento, usamos un bucle `for` durante k iteraciones.


In [ ]:
# Inicializar arreglo de ceros con la misma forma que 'targets'
# all_predictions = arreglo_de_ceros(forma_de targets)

# Iterar sobre cada fold del esquema de validación cruzada
# PARA CADA (indices_entrenamiento, indices_prueba) EN kf.split(features):

    # Separar datos según índices del fold actual
    # features_entrenamiento = features[indices_entrenamiento]
    # features_prueba        = features[indices_prueba]
    # targets_entrenamiento  = targets[indices_entrenamiento]
    # targets_prueba         = targets[indices_prueba]

    # Entrenar el árbol de decisión con el subconjunto de entrenamiento
    # d_tree.ajustar(features_entrenamiento, targets_entrenamiento)

    # Predecir sobre el subconjunto de prueba
    # predicciones = d_tree.predecir(features_prueba)

    # Almacenar predicciones en las posiciones correspondientes
    # all_predictions[indices_prueba] = predicciones

# FIN ciclo FOR

# Evaluar el modelo: mediana de los errores absolutos
# eval_d_tree = mediana( |all_predictions - targets| )


Ahora podemos visualizar las predicciones frente a los objetivos para evaluar el modelo.

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

# plot the results to see how well our model looks
plt.figure(figsize=(7,7))
plt.scatter(targets, all_predictions, s=0.4)
plt.xlim((0, targets.max()))
plt.ylim((0, predictions.max()))
plt.xlabel('Measured Redshift')
plt.ylabel('Predicted Redshift')
plt.show()

Cabe destacar que este gráfico muestra que muchas de las predicciones reproducen los valores objetivo, pero también hay muchos puntos que se desvían de la tendencia principal (valores atípicos).

--- 
### Clase espectral

Ahora, incorporaremos la característica 'spec_class' que, para cada muestra, puede tener uno de los valores posibles b'GALAXY' y b'QSO'.

Utilizando el array de variables 'classes' definido anteriormente, filtraremos el conjunto de datos para obtener un subconjunto de objetos con la clase 'GALAXY' y un subconjunto con los objetos del tipo 'QSO'.

In [ ]:
# QSO ids donde classes == QSO
# Galaxy ids donde classes == Galaxy

Por lo tanto, tenemos 8525 objetos QSO y 41475 objetos GALAXY.

Ahora, evaluaremos el árbol de decisión específicamente para cada tipo de objeto:

In [ ]:
# all_predictions[QSO_ids]

In [ ]:
# calcular eval_qso con mediana( |all_predictions[QSO] - targets[QSO]| )
# calcular eval_galaxia con mediana( |all_predictions[galaxia] - targets[galaxia]| )

print('The median of the residues for each type of objects is:\n')
print('QSOs : ', eval_QSO)
print('GALAXYs : ', eval_GALAXY)

Cabe destacar que se obtienen mejores resultados para las galaxias que para los cuásares. La razón es sencilla: los cuásares son objetos con un alto corrimiento al rojo y, por lo tanto, son más difíciles de observar, lo que resulta en una mayor dispersión de los resultados (hay más valores atípicos en el gráfico de predicciones frente a objetivos).

Para ilustrar el comportamiento del modelo para cada tipo de objeto, representamos nuevamente las predicciones frente a los objetivos, indicando la clase de la muestra mediante colores:

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

# plot the results to see how well our model looks
plt.figure(figsize=(7,7))
plt.scatter(targets[QSO_ids], all_predictions[QSO_ids], s=0.4, color='crimson', label='QSO')
plt.scatter(targets[GALAXY_ids], all_predictions[GALAXY_ids], s=0.4, color='steelblue', label='GALAXY')
plt.xlim((0, targets.max()))
plt.ylim((0, predictions.max()))
plt.xlabel('Measured Redshift')
plt.ylabel('Predicted Redshift')
plt.legend()
plt.show()